CIFAR ResNet-18. Helpers are loaded by the entry notebooks; do not run separately.

In [ ]:
def make_model(dummy_count=0):
    model = models.resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3,64,3,stride=1,padding=1,bias=False)
    nn.init.kaiming_normal_(model.conv1.weight,mode='fan_out',nonlinearity='relu')
    model.maxpool = nn.Identity()
    if dummy_count: model.dummy = nn.Linear(512,dummy_count)
    return model

def pre_features(model,x):
    return model.layer2(model.layer1(model.maxpool(model.relu(model.bn1(model.conv1(x))))))

def post_features(model,h):
    return model.avgpool(model.layer4(model.layer3(h))).flatten(1)

def forward_outputs(model,x):
    f=post_features(model,pre_features(model,x))
    z=model.fc(f)
    d=model.dummy(f) if hasattr(model,'dummy') else z.new_empty((len(z),0))
    return f,z,d